<a href="https://colab.research.google.com/github/Shineii86/PullShark/blob/main/notebooks/PullShark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/PullShark/refs/heads/main/images/PullShark.png" width="280px" alt="Pull Shark Achievement">
  <h1>🦈 GitHub Pull Shark Automator</h1>
  <p><b>Automate pull request creation and merging to unlock the Pull Shark achievement.</b></p>
  <p>
    <img src="https://img.shields.io/badge/Python-3.8%2B-3776AB?logo=python&logoColor=white" alt="Python">
    <img src="https://img.shields.io/badge/License-MIT-yellow" alt="License">
    <img src="https://img.shields.io/badge/PRs-Automated-28a745" alt="Automated">
  </p>
</div>

---

In [ ]:
#@title 🎨 Apply Dark/Light Mode Styling
#@markdown <small>Auto-detects your Colab theme and applies matching colors. Run this cell once.</small>

from IPython.display import display, HTML

display(HTML('''
<style>
  .ps-alert { padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif; border-left: 4px solid; }
  .ps-info { background-color: #e8f4fd; border-color: #2196F3; }
  .ps-success { background-color: #d4edda; border-color: #28a745; }
  .ps-warning { background-color: #fff3cd; border-color: #ffc107; }
  .ps-error { background-color: #f8d7da; border-color: #dc3545; }
  .ps-card { background-color: #f8f9fa; border: 1px solid #dee2e6; border-radius: 6px; padding: 14px 18px; margin: 8px 0; font-family: sans-serif; }
  .ps-card table td:first-child { color: #666; }
  @media (prefers-color-scheme: dark) {
    .ps-info { background-color: #1a2a3a; border-color: #4a9eff; }
    .ps-success { background-color: #1a2e1a; border-color: #4caf50; }
    .ps-warning { background-color: #2e2a1a; border-color: #ff9800; }
    .ps-error { background-color: #2e1a1a; border-color: #f44336; }
    .ps-card { background-color: #1e1e1e; border-color: #444; }
    .ps-card table td:first-child { color: #aaa; }
  }
</style>
'''))
print('✅ Styling applied — dark and light mode supported.')

<table>
  <tr>
    <td>
      <h3>📋 How It Works</h3>
      <ol>
        <li>Creates a <b>new branch</b> from your base branch</li>
        <li>Makes a <b>small commit</b> (updates README.md)</li>
        <li>Opens a <b>pull request</b> to the base branch</li>
        <li><b>Merges</b> the PR automatically</li>
        <li>Repeats for the number of PRs you specify</li>
      </ol>
    </td>
    <td>
      <h3>🦈 Pull Shark Tiers</h3>
      <table>
        <tr><td>🥉 <b>Default</b></td><td>2 PRs</td></tr>
        <tr><td>🥈 <b>Bronze</b></td><td>16 PRs</td></tr>
        <tr><td>🥇 <b>Silver</b></td><td>128 PRs</td></tr>
        <tr><td>🏆 <b>Gold</b></td><td>1024 PRs</td></tr>
      </table>
    </td>
  </tr>
</table>

<div class="ps-alert ps-warning">
  <b>⚠️ Important — Read Before Running</b>
  <ul style="margin: 8px 0 0 0;">
    <li>This script creates and merges <b>real pull requests</b> on your GitHub account.</li>
    <li><b>Never share your Personal Access Token</b> — treat it like a password.</li>
    <li>Use a <b>dedicated test repository</b> to avoid cluttering important projects.</li>
    <li>GitHub may rate-limit excessive API calls; the built-in delay helps prevent this.</li>
  </ul>
</div>

In [ ]:
#@title 📦 Step 1 — Install Dependencies & Load Package
#@markdown <small>Installs PyGithub and loads the PullShark package. Run this cell first.</small>

import sys, os, time
from IPython.display import display, HTML

display(HTML('<div class="ps-alert ps-info">⏳ Installing dependencies and loading PullShark...</div>'))

!pip install -q PyGithub

if os.path.exists('pullshark'):
    sys.path.insert(0, '.')
else:
    !git clone -q https://github.com/Shineii86/PullShark.git /tmp/pullshark-repo
    sys.path.insert(0, '/tmp/pullshark-repo')

from pullshark.config import Config
from pullshark.core import PullSharkBot
from pullshark import __version__

display(HTML(f'<div class="ps-alert ps-success">✅ <b>PullShark v{__version__}</b> loaded successfully!</div>'))

In [ ]:
#@title 🔌 Step 2 — Test Connection
#@markdown <small>Validates your token, repo access, API quota, and existing branches.</small>

TEST_TOKEN = ""  #@param {type:"string"}
TEST_USERNAME = ""  #@param {type:"string"}
TEST_REPO = ""  #@param {type:"string"}

from github import Github
from IPython.display import display, HTML

def check(label, ok, detail=''):
    color = '#28a745' if ok else '#dc3545'
    extra = f' <span style="color: #888;">— {detail}</span>' if detail else ''
    icon = '✅' if ok else '❌'
    display(HTML(f'<div style="font-family: monospace; padding: 2px 0; color: {color};">{icon} {label}{extra}</div>'))

if not all([TEST_TOKEN, TEST_USERNAME, TEST_REPO]):
    display(HTML('<div class="ps-alert ps-warning">⚠️ Fill in all three fields above and run this cell.</div>'))
else:
    display(HTML('<div style="font-family: sans-serif; padding: 4px 0;"><b>🔌 Testing connection...</b></div>'))
    token_ok = False
    repo_ok = False
    try:
        g = Github(TEST_TOKEN)
        actual_username = g.get_user().login
        check('Token valid', True, f'Authenticated as @{actual_username}')
        token_ok = True
    except Exception as e:
        check('Token valid', False, str(e)[:80])

    if token_ok:
        check('Username matches token', actual_username.lower() == TEST_USERNAME.lower(),
              f'Expected @{TEST_USERNAME}, got @{actual_username}' if actual_username.lower() != TEST_USERNAME.lower() else '')

    if token_ok:
        try:
            repo = g.get_user().get_repo(TEST_REPO)
            check('Repository accessible', True, f'{repo.full_name} ({"private" if repo.private else "public"})')
            repo_ok = True
        except Exception as e:
            check('Repository accessible', False, str(e)[:80])

    if token_ok and repo_ok:
        try:
            sha = repo.get_branch(repo.default_branch).commit.sha
            ref = repo.create_git_ref('refs/heads/_pullshark_test_', sha)
            ref.delete()
            check('Write access confirmed', True)
        except Exception as e:
            check('Write access confirmed', False, str(e)[:80])

    if token_ok:
        rate = g.get_rate_limit().core
        check('API rate limit', rate.remaining > 20, f'{rate.remaining}/{rate.limit} remaining (resets {rate.reset.strftime("%H:%M:%S")})')

    if token_ok and repo_ok:
        branches = [b.name for b in repo.get_branches() if b.name.startswith('auto-pr-')]
        if branches:
            check('Existing auto-pr branches', False, f'{len(branches)} found — run Step 5 to clean up')
        else:
            check('Existing auto-pr branches', True, 'None — repo is clean')

    if token_ok and repo_ok:
        display(HTML('<div class="ps-alert ps-success"><b>🎉 All checks passed!</b> Ready for Step 3.</div>'))
    else:
        display(HTML('<div class="ps-alert ps-error"><b>⚠️ Some checks failed.</b> Fix issues above before proceeding.</div>'))

In [ ]:
#@title 🔍 Step 3 — Dry Run (Preview)
#@markdown <small>Preview what the bot will do without making changes. <b>Auto-fills from Step 2.</b></small>

# Auto-fill from Step 2
try:
    _auto_token = TEST_TOKEN
    _auto_user = TEST_USERNAME
    _auto_repo = TEST_REPO
    _auto_from = 'Step 2'
except NameError:
    _auto_token = 'ghp_xxxxxxxxxxxxxxxxxxxx'
    _auto_user = 'Shineii86'
    _auto_repo = 'PullShark'
    _auto_from = ''

DRY_USERNAME = _auto_user  #@param {type:"string"}
DRY_TOKEN = _auto_token  #@param {type:"string"}
DRY_REPO = _auto_repo  #@param {type:"string"}
DRY_NUM_PRS = 4  #@param {type:"slider", min:2, max:50, step:1}
DRY_MERGE_METHOD = "merge"  #@param ["merge", "squash", "rebase"]

from IPython.display import display, HTML

if _auto_from:
    display(HTML(f'<div class="ps-alert ps-success">🔗 <b>Auto-detected credentials from {_auto_from}.</b> Edit fields to override.</div>'))

display(HTML('<div class="ps-alert ps-info">🔍 <b>Dry Run Mode</b> — no changes will be made.</div>'))

config = Config(github_username=DRY_USERNAME, github_token=DRY_TOKEN, repo_name=DRY_REPO,
                num_prs=DRY_NUM_PRS, dry_run=True, merge_method=DRY_MERGE_METHOD)

errors = config.validate()
if errors:
    display(HTML('<div class="ps-alert ps-error"><b>❌ Errors:</b><ul style="margin:6px 0 0 0;">' + ''.join(f'<li>{e}</li>' for e in errors) + '</ul></div>'))
else:
    PullSharkBot(config).run()
    display(HTML('<div class="ps-alert ps-success">✅ <b>Dry run complete!</b> Proceed to Step 4 if everything looks right.</div>'))

In [ ]:
#@title 🚀 Step 4 — Run for Real
#@markdown <small>Create and merge PRs. <b>Auto-fills from Step 3.</b> This makes real changes!</small>

# Auto-fill from Step 3
try:
    _auto_token = DRY_TOKEN
    _auto_user = DRY_USERNAME
    _auto_repo = DRY_REPO
    _auto_method = DRY_MERGE_METHOD
    _auto_from = 'Step 3'
except NameError:
    _auto_token = 'ghp_xxxxxxxxxxxxxxxxxxxx'
    _auto_user = 'Shineii86'
    _auto_repo = 'PullShark'
    _auto_method = 'merge'
    _auto_from = ''

#@markdown ### GitHub Credentials
GITHUB_USERNAME = _auto_user  #@param {type:"string"}
GITHUB_TOKEN = _auto_token  #@param {type:"string"}

#@markdown ### Repository
REPO_NAME = _auto_repo  #@param {type:"string"}
BASE_BRANCH = "main"  #@param {type:"string"}

#@markdown ### Automation Settings
NUM_PRS = 4  #@param {type:"slider", min:2, max:50, step:1}
DELAY_SECONDS = 10  #@param {type:"slider", min:5, max:60, step:5}
MAX_RETRIES = 3  #@param {type:"slider", min:1, max:10, step:1}
MERGE_METHOD = _auto_method  #@param ["merge", "squash", "rebase"]

#@markdown ### Pre-flight
CHECK_RATE_LIMIT = True  #@param {type:"boolean"}

from IPython.display import display, HTML
import time as _time

if _auto_from:
    display(HTML(f'<div class="ps-alert ps-success">🔗 <b>Auto-detected credentials from {_auto_from}.</b> Edit fields to override.</div>'))

config = Config(github_username=GITHUB_USERNAME, github_token=GITHUB_TOKEN, repo_name=REPO_NAME,
                num_prs=NUM_PRS, base_branch=BASE_BRANCH, delay_seconds=DELAY_SECONDS,
                max_retries=MAX_RETRIES, merge_method=MERGE_METHOD)

errors = config.validate()
if errors:
    display(HTML('<div class="ps-alert ps-error"><b>❌ Errors:</b><ul style="margin:6px 0 0 0;">' + ''.join(f'<li>{e}</li>' for e in errors) + '</ul></div>'))
else:
    bot = PullSharkBot(config)

    if CHECK_RATE_LIMIT:
        try:
            info = bot.check_rate_limit()
            cls = 'ps-success' if info['enough_for_run'] else 'ps-warning'
            icon = '✅' if info['enough_for_run'] else '⚠️'
            display(HTML(f'<div class="ps-alert {cls}">{icon} <b>API Quota:</b> {info["remaining"]}/{info["limit"]} remaining (resets {info["reset"]})</div>'))
        except Exception as e:
            display(HTML(f'<div class="ps-alert ps-warning">⚠️ Could not check rate limit: {str(e)[:60]}</div>'))

    est = NUM_PRS * (DELAY_SECONDS + 8)
    display(HTML(f'''
    <div class="ps-card">
      <b>📋 Configuration Summary</b>
      <table style="margin-top: 8px; font-size: 14px;">
        <tr><td style="padding: 2px 12px 2px 0;">User</td><td><code>@{GITHUB_USERNAME}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Repository</td><td><code>{REPO_NAME}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Base Branch</td><td><code>{BASE_BRANCH}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Pull Requests</td><td><b>{NUM_PRS}</b></td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Merge Method</td><td><code>{MERGE_METHOD}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Delay</td><td>{DELAY_SECONDS}s between PRs</td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Max Retries</td><td>{MAX_RETRIES} per merge</td></tr>
        <tr><td style="padding: 2px 12px 2px 0;">Est. Time</td><td>~{est // 60}m {est % 60}s</td></tr>
      </table>
    </div>
    '''))

    display(HTML('<div class="ps-alert ps-info">🚀 <b>Starting automation...</b></div>'))
    start_time = _time.time()
    successful = bot.run()
    elapsed = int(_time.time() - start_time)
    mins, secs = divmod(elapsed, 60)

    success = successful >= 2
    cls = 'ps-success' if success else 'ps-error'
    icon = '🎉' if success else '⚠️'
    title = 'Achievement Unlocked!' if success else 'More PRs Needed'
    tier = 'Gold 🏆' if successful >= 1024 else 'Silver 🥇' if successful >= 128 else 'Bronze 🥈' if successful >= 16 else 'Default 🥉' if successful >= 2 else 'None'

    display(HTML(f'''
    <div class="ps-alert {cls}" style="border-width: 2px; padding: 16px 20px;">
      <div style="font-size: 20px; margin-bottom: 8px;">{icon} <b>{title}</b></div>
      <table style="font-size: 14px;">
        <tr><td style="padding: 2px 16px 2px 0;">Merged PRs</td><td><b>{successful}</b> / {NUM_PRS}</td></tr>
        <tr><td style="padding: 2px 16px 2px 0;">Pull Shark Tier</td><td><b>{tier}</b></td></tr>
        <tr><td style="padding: 2px 16px 2px 0;">Merge Method</td><td>{MERGE_METHOD}</td></tr>
        <tr><td style="padding: 2px 16px 2px 0;">Time Elapsed</td><td>{mins}m {secs}s</td></tr>
      </table>
    </div>
    '''))

    if success:
        display(HTML('<div class="ps-alert ps-warning">💡 <b>Next:</b> Check your <a href="https://github.com" target="_blank">GitHub profile</a> for the badge. 🧹 Run Step 5 to clean up branches.</div>'))

In [ ]:
#@title 🧹 Step 5 — Cleanup Branches
#@markdown <small>Delete auto-created branches. <b>Auto-detects credentials from Step 4</b> if already run.</small>

# Try to auto-detect from Step 4 variables
try:
    _auto_token = GITHUB_TOKEN
    _auto_user = GITHUB_USERNAME
    _auto_repo = REPO_NAME
    _auto_detected = True
except NameError:
    _auto_token = ""
    _auto_user = ""
    _auto_repo = ""
    _auto_detected = False

CLEAN_TOKEN = _auto_token  #@param {type:"string"}
CLEAN_USERNAME = _auto_user  #@param {type:"string"}
CLEAN_REPO = _auto_repo  #@param {type:"string"}
CLEAN_DRY_RUN = True  #@param {type:"boolean"}

from IPython.display import display, HTML

if _auto_detected:
    display(HTML('<div class="ps-alert ps-success">🔗 <b>Auto-detected credentials from Step 4.</b> Edit fields below to override.</div>'))

if not all([CLEAN_TOKEN, CLEAN_USERNAME, CLEAN_REPO]):
    display(HTML('<div class="ps-alert ps-warning">⚠️ Fill in all three fields above and run this cell.</div>'))
else:
    mode = 'PREVIEW (dry run)' if CLEAN_DRY_RUN else 'LIVE DELETION'
    cls = 'ps-info' if CLEAN_DRY_RUN else 'ps-error'
    display(HTML(f'<div class="ps-alert {cls}">🧹 <b>Branch Cleanup</b> — mode: <b>{mode}</b></div>'))

    config = Config(github_username=CLEAN_USERNAME, github_token=CLEAN_TOKEN, repo_name=CLEAN_REPO)
    deleted = PullSharkBot(config).clean(dry_run=CLEAN_DRY_RUN)

    if deleted == 0:
        display(HTML('<div class="ps-alert ps-success">✨ <b>Repo is clean!</b> No matching branches found.</div>'))
    elif CLEAN_DRY_RUN:
        display(HTML(f'<div class="ps-alert ps-info">📋 <b>{deleted} branch(es) would be deleted.</b> Uncheck dry-run and re-run to delete.</div>'))
    else:
        display(HTML(f'<div class="ps-alert ps-success">🗑️ <b>{deleted} branch(es) deleted!</b> Repo is clean.</div>'))

---

<details>
<summary><b>📚 Troubleshooting & Help</b></summary>

| Issue | Solution |
|:------|:---------|
| `BadCredentialsException` | Token is wrong or expired. Generate a new one. |
| `405 Not mergeable` | Enable **Allow auto-merge** in repo Settings → General → Pull Requests. |
| Hangs at "Waiting for mergeability" | PR may have a conflict. Delete the branch manually and retry. |
| `RateLimitExceededException` | Wait an hour or increase `DELAY_SECONDS`. |
| No badge after success | Wait a few minutes and refresh your GitHub profile. |

</details>

<details>
<summary><b>🔐 How to Get a Personal Access Token</b></summary>

**Classic Token (Recommended for beginners):**
1. Go to **Settings** → **Developer settings** → **Personal access tokens** → **Tokens (classic)**.
2. Click **Generate new token (classic)**.
3. Name it (e.g., `Pull Shark Bot`).
4. Check the **`repo`** scope.
5. Click **Generate token** and copy it immediately.

**Fine-Grained Token (More secure, recommended):**
1. Go to **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens**.
2. Click **Generate new token**.
3. Set **Resource owner** to your username.
4. Under **Repository access**, select **Only select repositories** → pick your target repo.
5. Under **Permissions** → **Repository permissions**, set:
   - **Contents**: `Read and write`
   - **Metadata**: `Read-only`
   - **Pull requests**: `Read and write`
6. Click **Generate token** and copy it.

</details>

<details>
<summary><b>🔀 Merge Methods Explained</b></summary>

| Method | What it does |
|:-------|:-------------|
| `merge` | Creates a merge commit. Preserves all branch history. Default behavior. |
| `squash` | Combines all commits into one. Cleaner history, single commit per PR. |
| `rebase` | Replays commits on base branch. Linear history, no merge commit. |

</details>

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for GitHub achievement hunters*

***Found this useful? Give it a Star! ⭐ [Shineii86/PullShark](https://github.com/Shineii86/PullShark)***

</div>